## Raw Data and Balanced Data with: 
Raw Data, SSO, SMOTE, ClusterSMOTE, LORAS, SMOTETomek, ROS
### ENN Is created befor.
#### --------------------------------------------------------
## Models:
RF, DT, XGBost, CatBoost    


#### ----------------------------------------------
# Data Analysis

In [1]:
import scanpy as sc

# Load the h5ad file
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# Print basic information about the dataset
print("=== Basic Info ===")
print(adata)

# Show the shape of the data (cells x genes)
print("\n=== Shape ===")
print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")

# Show observation (cell metadata)
print("\n=== Observations (adata.obs) ===")
print(adata.obs.head())
print("\nColumns in adata.obs:")
print(adata.obs.columns)

# Show variable (gene metadata)
print("\n=== Variables (adata.var) ===")
print(adata.var.head())
print("\nColumns in adata.var:")
print(adata.var.columns)

# Show available layers (if any)
print("\n=== Layers ===")
print(adata.layers.keys())

# Show uns (unstructured data)
print("\n=== Unstructured data (adata.uns) ===")
print(adata.uns.keys())

# Show obsm (multi-dimensional annotations like PCA/UMAP)
print("\n=== obsm (embeddings) ===")
print(adata.obsm.keys())

# Show varm (gene-level embeddings)
print("\n=== varm ===")
print(adata.varm.keys())

# Check raw data (if exists)
print("\n=== Raw data ===")
print(adata.raw)

# Preview expression matrix (first few cells and genes)
print("\n=== Expression matrix preview ===")
print(adata.X[:5, :5])

=== Basic Info ===
AnnData object with n_obs × n_vars = 2463 × 17505
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'cell_type', 'complexity', 'umap1', 'umap2', 'g1s_score', 'g2m_score', 'cell_cycle_phase', 'mp_top_score', 'mp_top', 'mp_assignment', 'disease', 'nCount_tmp', 'nFeature_tmp', 'percent.ribo', 'percent.mito', 'log10GenesPerUmi', 'ident'
    uns: 'X_name'
    obsm: 'PCA', 'UMAP'
    layers: 'logcounts'

=== Shape ===
Number of cells: 2463
Number of genes: 17505

=== Observations (adata.obs) ===
                                                               orig.ident  \
AAACCTGAGATAGCAT-3,C26,Endothelial,1247,-16.72,...  Ma2019_Liver-Biliary_   
AAACCTGAGATGTAAC-9,H38,Malignant,3524,29.304,15...  Ma2019_Liver-Biliary_   
AAACCTGAGGAATTAC-9,H38,Fibroblast,1562,10.737,-...  Ma2019_Liver-Biliary_   
AAACCTGAGGTACTCT-22,C66,Malignant,2446,14.063,-...  Ma2019_Liver-Biliary_   
AAACCTGCAATGGACG-13,H37,Malignant,4797,6.3469,4...  Ma2019_Liver-Biliary_   

          

#### --------------------------------------------------------------
# Raw Data Modeling

In [2]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# =========================
# Plot settings (high quality + Times font)
# =========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['figure.dpi'] = 300

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# =========================
# Feature matrix (use logcounts if available)
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int)
y = np.array(y, dtype=int)  # ensure compatibility with CatBoost

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Models (default settings)
# =========================
models = {
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# Train & Evaluate
# =========================
for name, model in models.items():
    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Classification Report
    # =========================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # =========================
    # Confusion Matrix
    # =========================
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-Malignant', 'Malignant'],
                yticklabels=['Non-Malignant', 'Malignant'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{name}_confusion_matrix.png", bbox_inches='tight')
    plt.close()

    # =========================
    # ROC Curve & AUC
    # =========================
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.savefig(f"{name}_ROC.png", bbox_inches='tight')
    plt.close()

    print(f"AUC: {roc_auc:.4f}")


========== RandomForest ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       293

    accuracy                           1.00       493
   macro avg       1.00      1.00      1.00       493
weighted avg       1.00      1.00      1.00       493

AUC: 1.0000

========== DecisionTree ==========

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       200
           1       0.98      0.99      0.99       293

    accuracy                           0.99       493
   macro avg       0.99      0.98      0.99       493
weighted avg       0.99      0.99      0.99       493

AUC: 0.9841

========== XGBoost ==========


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:08:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       293

    accuracy                           1.00       493
   macro avg       1.00      1.00      1.00       493
weighted avg       1.00      1.00      1.00       493

AUC: 1.0000

========== CatBoost ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       293

    accuracy                           1.00       493
   macro avg       1.00      1.00      1.00       493
weighted avg       1.00      1.00      1.00       493

AUC: 1.0000


#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------

# SSO Data

In [3]:
import scanpy as sc
import numpy as np

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# =========================
# Feature matrix
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int).values

# =========================
# Find class indices
# =========================
idx_0 = np.where(y == 0)[0]
idx_1 = np.where(y == 1)[0]

print("Before balancing:")
print(f"Non-Malignant: {len(idx_0)}")
print(f"Malignant: {len(idx_1)}")

# =========================
# Identify minority
# =========================
if len(idx_0) > len(idx_1):
    majority_idx = idx_0
    minority_idx = idx_1
    minority_label = 1
else:
    majority_idx = idx_1
    minority_idx = idx_0
    minority_label = 0

X_min = X[minority_idx]

# =========================
# SSO parameters
# =========================
n_to_generate = len(majority_idx) - len(minority_idx)
step_size = 0.02   # movement step (important)
iterations = 5     # SSO iterations

# =========================
# SSO-based sample generation
# =========================
synthetic_samples = []

for i in range(n_to_generate):
    # randomly pick a base sample
    idx = np.random.randint(0, X_min.shape[0])
    shark = X_min[idx].copy()

    # SSO movement
    for _ in range(iterations):
        direction = np.random.normal(0, 1, size=shark.shape)
        shark = shark + step_size * direction  # movement

    synthetic_samples.append(shark)

X_syn = np.array(synthetic_samples)

# =========================
# Combine with original data
# =========================
X_new = np.vstack([X, X_syn])
y_new = np.concatenate([y, np.full(n_to_generate, minority_label)])

# =========================
# Create new AnnData (KEEP metadata!)
# =========================
import pandas as pd
from scipy.sparse import csr_matrix

adata_new = sc.AnnData(X=csr_matrix(X_new))

# keep original obs
obs_original = adata.obs.copy()

# create obs for synthetic samples
obs_syn = pd.DataFrame(index=range(n_to_generate))
obs_syn['cell_type'] = 'Malignant' if minority_label == 1 else 'Non-Malignant'

# fill other columns with NaN to keep structure
for col in obs_original.columns:
    if col not in obs_syn.columns:
        obs_syn[col] = np.nan

# combine
adata_new.obs = pd.concat([obs_original, obs_syn], ignore_index=True)

# =========================
# Check
# =========================
y_check = (adata_new.obs['cell_type'] == 'Malignant').astype(int).values

print("\nAfter balancing:")
print(f"Non-Malignant: {np.sum(y_check == 0)}")
print(f"Malignant: {np.sum(y_check == 1)}")

# =========================
# Save
# =========================
adata_new.write("Ma2019_balanced_SSO.h5ad")

print("\nSaved: Ma2019_balanced_SSO.h5ad")

Before balancing:
Non-Malignant: 1001
Malignant: 1462

After balancing:
Non-Malignant: 1462
Malignant: 1462

Saved: Ma2019_balanced_SSO.h5ad


C:\Users\Arman\AppData\Local\Temp\ipykernel_10536\3626956513.py:101: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  adata_new.obs = pd.concat([obs_original, obs_syn], ignore_index=True)
C:\Users\Arman\AppData\Local\Temp\ipykernel_10536\3626956513.py:101: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  adata_new.obs = pd.concat([obs_original, obs_syn], ignore_index=True)
C:\Users\Arman\AppData\Local\Temp\ipykernel_10536\3626956513.py:101: FutureWarning: The behavior of DataFrame concatenation with e

# Modeling for SSO Data

In [4]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# =========================
# Plot settings (high quality + Times font)
# =========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['figure.dpi'] = 300

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_balanced_SSO.h5ad")

# =========================
# Feature matrix (use logcounts if available)
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int)
y = np.array(y, dtype=int)  # ensure compatibility with CatBoost

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Models (default settings)
# =========================
models = {
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# Train & Evaluate
# =========================
for name, model in models.items():
    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Classification Report
    # =========================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # =========================
    # Confusion Matrix
    # =========================
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-Malignant', 'Malignant'],
                yticklabels=['Non-Malignant', 'Malignant'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{name}_confusion_matrix.png", bbox_inches='tight')
    plt.close()

    # =========================
    # ROC Curve & AUC
    # =========================
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.savefig(f"{name}_ROC.png", bbox_inches='tight')
    plt.close()

    print(f"AUC: {roc_auc:.4f}")


========== RandomForest ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9996

========== DecisionTree ==========

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       293
           1       0.98      0.99      0.99       292

    accuracy                           0.99       585
   macro avg       0.99      0.99      0.99       585
weighted avg       0.99      0.99      0.99       585

AUC: 0.9863

========== XGBoost ==========


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9997

========== CatBoost ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9995


#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------

# SMOTE Data

In [5]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from imblearn.over_sampling import SMOTE

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# =========================
# Feature matrix
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int).values

print("Before SMOTE:")
print(f"Non-Malignant: {np.sum(y==0)}")
print(f"Malignant: {np.sum(y==1)}")

# =========================
# Apply SMOTE
# =========================
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print("\nAfter SMOTE:")
print(f"Non-Malignant: {np.sum(y_res==0)}")
print(f"Malignant: {np.sum(y_res==1)}")

# =========================
# Create new AnnData (KEEP original structure)
# =========================
adata_smote = sc.AnnData(X=csr_matrix(X_res))

# =========================
# Build obs safely
# =========================
obs = pd.DataFrame(index=range(len(y_res)))

obs["cell_type"] = np.where(y_res == 1, "Malignant", "Non-Malignant")

# keep original metadata columns structure (fill NaN for synthetic)
for col in adata.obs.columns:
    if col != "cell_type":
        obs[col] = np.nan

adata_smote.obs = obs

# =========================
# Save dataset
# =========================
adata_smote.write("Ma2019_SMOTE_balanced.h5ad")

print("\nSaved: Ma2019_SMOTE_balanced.h5ad")

Before SMOTE:
Non-Malignant: 1001
Malignant: 1462

After SMOTE:
Non-Malignant: 1462
Malignant: 1462

Saved: Ma2019_SMOTE_balanced.h5ad


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


# Modeling for SMOTE Data

In [6]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# =========================
# Plot settings (high quality + Times font)
# =========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['figure.dpi'] = 300

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_SMOTE_balanced.h5ad")

# =========================
# Feature matrix (use logcounts if available)
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int)
y = np.array(y, dtype=int)  # ensure compatibility with CatBoost

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Models (default settings)
# =========================
models = {
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# Train & Evaluate
# =========================
for name, model in models.items():
    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Classification Report
    # =========================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # =========================
    # Confusion Matrix
    # =========================
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-Malignant', 'Malignant'],
                yticklabels=['Non-Malignant', 'Malignant'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{name}_confusion_matrix.png", bbox_inches='tight')
    plt.close()

    # =========================
    # ROC Curve & AUC
    # =========================
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.savefig(f"{name}_ROC.png", bbox_inches='tight')
    plt.close()

    print(f"AUC: {roc_auc:.4f}")


========== RandomForest ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 1.0000

========== DecisionTree ==========

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       293
           1       0.98      0.98      0.98       292

    accuracy                           0.98       585
   macro avg       0.98      0.98      0.98       585
weighted avg       0.98      0.98      0.98       585

AUC: 0.9778

========== XGBoost ==========


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:23:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9999

========== CatBoost ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9997


#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------

# Cluster SMOTE Data

In [8]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# =========================
# Feature matrix
# =========================
X = adata.layers["logcounts"] if "logcounts" in adata.layers else adata.X
X = X.toarray()

# =========================
# Labels
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int).values

# =========================
# Clustering
# =========================
n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

X_res_all = []
y_res_all = []

# =========================
# Cluster-wise SMOTE (FIXED)
# =========================
for c in range(n_clusters):
    idx = np.where(clusters == c)[0]

    X_c = X[idx]
    y_c = y[idx]

    # Need both classes in cluster
    if len(np.unique(y_c)) < 2:
        X_res_all.append(X_c)
        y_res_all.append(y_c)
        continue

    # Count minority class size
    min_class_count = np.min(np.bincount(y_c))

    # FIX: adjust k_neighbors safely
    k_neighbors = max(1, min(5, min_class_count - 1))

    # If not enough samples, skip SMOTE
    if min_class_count < 2:
        X_res_all.append(X_c)
        y_res_all.append(y_c)
        continue

    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)

    X_res_c, y_res_c = smote.fit_resample(X_c, y_c)

    X_res_all.append(X_res_c)
    y_res_all.append(y_res_c)

# =========================
# Merge results
# =========================
X_res = np.vstack(X_res_all)
y_res = np.concatenate(y_res_all)

print("After Cluster-SMOTE:")
print(f"Non-Malignant: {np.sum(y_res==0)}")
print(f"Malignant: {np.sum(y_res==1)}")

# =========================
# Save (keep structure minimal but safe)
# =========================
adata_balanced = sc.AnnData(X=csr_matrix(X_res))

adata_balanced.obs = pd.DataFrame({
    "cell_type": np.where(y_res == 1, "Malignant", "Non-Malignant")
})

adata_balanced.write("Ma2019_ClusterSMOTE_fixed.h5ad")

print("Saved: Ma2019_ClusterSMOTE_fixed.h5ad")

After Cluster-SMOTE:
Non-Malignant: 1628
Malignant: 1462
Saved: Ma2019_ClusterSMOTE_fixed.h5ad


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


# Modeling for ClusterSMOTE Data

In [9]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# =========================
# Plot settings (high quality + Times font)
# =========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['figure.dpi'] = 300

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_ClusterSMOTE_fixed.h5ad")

# =========================
# Feature matrix (use logcounts if available)
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int)
y = np.array(y, dtype=int)  # ensure compatibility with CatBoost

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Models (default settings)
# =========================
models = {
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# Train & Evaluate
# =========================
for name, model in models.items():
    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Classification Report
    # =========================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # =========================
    # Confusion Matrix
    # =========================
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-Malignant', 'Malignant'],
                yticklabels=['Non-Malignant', 'Malignant'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{name}_confusion_matrix.png", bbox_inches='tight')
    plt.close()

    # =========================
    # ROC Curve & AUC
    # =========================
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.savefig(f"{name}_ROC.png", bbox_inches='tight')
    plt.close()

    print(f"AUC: {roc_auc:.4f}")


========== RandomForest ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       326
           1       1.00      1.00      1.00       292

    accuracy                           1.00       618
   macro avg       1.00      1.00      1.00       618
weighted avg       1.00      1.00      1.00       618

AUC: 1.0000

========== DecisionTree ==========

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       326
           1       0.98      0.97      0.97       292

    accuracy                           0.98       618
   macro avg       0.98      0.98      0.98       618
weighted avg       0.98      0.98      0.98       618

AUC: 0.9754

========== XGBoost ==========


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:39:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       326
           1       1.00      1.00      1.00       292

    accuracy                           1.00       618
   macro avg       1.00      1.00      1.00       618
weighted avg       1.00      1.00      1.00       618

AUC: 1.0000

========== CatBoost ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       326
           1       1.00      1.00      1.00       292

    accuracy                           1.00       618
   macro avg       1.00      1.00      1.00       618
weighted avg       1.00      1.00      1.00       618

AUC: 1.0000


#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------

# LORAS Data

In [10]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# =========================
# Feature matrix
# =========================
X = adata.layers["logcounts"] if "logcounts" in adata.layers else adata.X
X = X.toarray()

# =========================
# Labels
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int).values

print("Before LoRAS:")
print(f"Non-Malignant: {np.sum(y==0)}")
print(f"Malignant: {np.sum(y==1)}")

# =========================
# Identify minority class
# =========================
minority_label = 1 if np.sum(y==1) < np.sum(y==0) else 0
minority_idx = np.where(y == minority_label)[0]

X_min = X[minority_idx]

# =========================
# LoRAS parameters
# =========================
k = 5  # neighbors
n_to_generate = abs(np.sum(y==0) - np.sum(y==1))

# Fit nearest neighbors on minority class
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_min)

synthetic_samples = []

# =========================
# LoRAS generation
# =========================
for _ in range(n_to_generate):
    # pick random minority point
    i = np.random.randint(0, X_min.shape[0])
    x = X_min[i]

    # get neighbors
    neighbors_idx = nn.kneighbors([x], return_distance=False)[0]
    neighbors = X_min[neighbors_idx]

    # create shadow samples (add small noise)
    shadows = neighbors + np.random.normal(0, 0.01, neighbors.shape)

    # affine combination (random weights)
    weights = np.random.dirichlet(np.ones(k))
    synthetic = np.sum(shadows * weights[:, None], axis=0)

    synthetic_samples.append(synthetic)

X_syn = np.array(synthetic_samples)
y_syn = np.full(n_to_generate, minority_label)

# =========================
# Combine
# =========================
X_res = np.vstack([X, X_syn])
y_res = np.concatenate([y, y_syn])

print("\nAfter LoRAS:")
print(f"Non-Malignant: {np.sum(y_res==0)}")
print(f"Malignant: {np.sum(y_res==1)}")

# =========================
# Create AnnData (preserve minimal structure)
# =========================
adata_balanced = sc.AnnData(X=csr_matrix(X_res))

adata_balanced.obs = pd.DataFrame({
    "cell_type": np.where(y_res == 1, "Malignant", "Non-Malignant")
})

# =========================
# Save
# =========================
adata_balanced.write("Ma2019_LoRAS_balanced.h5ad")

print("\nSaved: Ma2019_LoRAS_balanced.h5ad")

Before LoRAS:
Non-Malignant: 1001
Malignant: 1462

After LoRAS:
Non-Malignant: 1462
Malignant: 1462

Saved: Ma2019_LoRAS_balanced.h5ad


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


# Modeling for LORAS Data

In [11]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# =========================
# Plot settings (high quality + Times font)
# =========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['figure.dpi'] = 300

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_LoRAS_balanced.h5ad")

# =========================
# Feature matrix (use logcounts if available)
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int)
y = np.array(y, dtype=int)  # ensure compatibility with CatBoost

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Models (default settings)
# =========================
models = {
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# Train & Evaluate
# =========================
for name, model in models.items():
    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Classification Report
    # =========================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # =========================
    # Confusion Matrix
    # =========================
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-Malignant', 'Malignant'],
                yticklabels=['Non-Malignant', 'Malignant'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{name}_confusion_matrix.png", bbox_inches='tight')
    plt.close()

    # =========================
    # ROC Curve & AUC
    # =========================
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.savefig(f"{name}_ROC.png", bbox_inches='tight')
    plt.close()

    print(f"AUC: {roc_auc:.4f}")


========== RandomForest ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9999

========== DecisionTree ==========

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       293
           1       0.98      0.97      0.98       292

    accuracy                           0.98       585
   macro avg       0.98      0.98      0.98       585
weighted avg       0.98      0.98      0.98       585

AUC: 0.9778

========== XGBoost ==========


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:48:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9998

========== CatBoost ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9995


#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------

# SMOTETomek Data

In [12]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from imblearn.combine import SMOTETomek

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# =========================
# Feature matrix
# =========================
X = adata.layers["logcounts"] if "logcounts" in adata.layers else adata.X
X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int).values

print("Before SMOTETomek:")
print(f"Non-Malignant: {np.sum(y==0)}")
print(f"Malignant: {np.sum(y==1)}")

# =========================
# Apply SMOTETomek
# =========================
smote_tomek = SMOTETomek(random_state=42)

X_res, y_res = smote_tomek.fit_resample(X, y)

print("\nAfter SMOTETomek:")
print(f"Non-Malignant: {np.sum(y_res==0)}")
print(f"Malignant: {np.sum(y_res==1)}")

# =========================
# Create AnnData (keep structure minimal)
# =========================
adata_balanced = sc.AnnData(X=csr_matrix(X_res))

adata_balanced.obs = pd.DataFrame({
    "cell_type": np.where(y_res == 1, "Malignant", "Non-Malignant")
})

# =========================
# Save dataset
# =========================
adata_balanced.write("Ma2019_SMOTETomek_balanced.h5ad")

print("\nSaved: Ma2019_SMOTETomek_balanced.h5ad")

Before SMOTETomek:
Non-Malignant: 1001
Malignant: 1462

After SMOTETomek:
Non-Malignant: 1462
Malignant: 1462

Saved: Ma2019_SMOTETomek_balanced.h5ad


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


# Modeling for SMOTETomek Data

In [13]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# =========================
# Plot settings (high quality + Times font)
# =========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['figure.dpi'] = 300

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_SMOTETomek_balanced.h5ad")

# =========================
# Feature matrix (use logcounts if available)
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int)
y = np.array(y, dtype=int)  # ensure compatibility with CatBoost

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Models (default settings)
# =========================
models = {
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# Train & Evaluate
# =========================
for name, model in models.items():
    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Classification Report
    # =========================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # =========================
    # Confusion Matrix
    # =========================
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-Malignant', 'Malignant'],
                yticklabels=['Non-Malignant', 'Malignant'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{name}_confusion_matrix.png", bbox_inches='tight')
    plt.close()

    # =========================
    # ROC Curve & AUC
    # =========================
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.savefig(f"{name}_ROC.png", bbox_inches='tight')
    plt.close()

    print(f"AUC: {roc_auc:.4f}")


========== RandomForest ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       293
           1       0.99      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 1.0000

========== DecisionTree ==========

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       293
           1       0.99      0.98      0.98       292

    accuracy                           0.98       585
   macro avg       0.98      0.98      0.98       585
weighted avg       0.98      0.98      0.98       585

AUC: 0.9846

========== XGBoost ==========


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:06:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9999

========== CatBoost ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 0.9997


#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------
#### ----------------------------------------------------------------

# ROS Data

In [14]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from imblearn.over_sampling import RandomOverSampler

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# =========================
# Feature matrix
# =========================
X = adata.layers["logcounts"] if "logcounts" in adata.layers else adata.X
X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int).values

print("Before ROS:")
print(f"Non-Malignant: {np.sum(y==0)}")
print(f"Malignant: {np.sum(y==1)}")

# =========================
# Apply Random Over Sampling
# =========================
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

print("\nAfter ROS:")
print(f"Non-Malignant: {np.sum(y_res==0)}")
print(f"Malignant: {np.sum(y_res==1)}")

# =========================
# Create new AnnData (keep structure minimal)
# =========================
adata_ros = sc.AnnData(X=csr_matrix(X_res))

adata_ros.obs = pd.DataFrame({
    "cell_type": np.where(y_res == 1, "Malignant", "Non-Malignant")
})

# =========================
# Save dataset
# =========================
adata_ros.write("Ma2019_ROS_balanced.h5ad")

print("\nSaved: Ma2019_ROS_balanced.h5ad")

Before ROS:
Non-Malignant: 1001
Malignant: 1462

After ROS:
Non-Malignant: 1462
Malignant: 1462

Saved: Ma2019_ROS_balanced.h5ad


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


# Modeling for ROS Data

In [15]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# =========================
# Plot settings (high quality + Times font)
# =========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['figure.dpi'] = 300

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_ROS_balanced.h5ad")

# =========================
# Feature matrix (use logcounts if available)
# =========================
if "logcounts" in adata.layers:
    X = adata.layers["logcounts"]
else:
    X = adata.X

X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int)
y = np.array(y, dtype=int)  # ensure compatibility with CatBoost

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Models (default settings)
# =========================
models = {
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# Train & Evaluate
# =========================
for name, model in models.items():
    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Classification Report
    # =========================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # =========================
    # Confusion Matrix
    # =========================
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-Malignant', 'Malignant'],
                yticklabels=['Non-Malignant', 'Malignant'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{name}_confusion_matrix.png", bbox_inches='tight')
    plt.close()

    # =========================
    # ROC Curve & AUC
    # =========================
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.savefig(f"{name}_ROC.png", bbox_inches='tight')
    plt.close()

    print(f"AUC: {roc_auc:.4f}")


========== RandomForest ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 1.0000

========== DecisionTree ==========

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       293
           1       1.00      0.97      0.99       292

    accuracy                           0.99       585
   macro avg       0.99      0.99      0.99       585
weighted avg       0.99      0.99      0.99       585

AUC: 0.9863

========== XGBoost ==========


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:12:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 1.0000

========== CatBoost ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       293
           1       1.00      1.00      1.00       292

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585

AUC: 1.0000


# ------------------------------
# ------------------------------
# ------------------------------
# ------------------------------
# ------------------------------
# ------------------------------

# Modeling for ENN Data

In [17]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# =========================
# Plot settings (high quality + Times font)
# =========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['figure.dpi'] = 300

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary_ENN_Balanced.h5ad")

# =========================
# Feature matrix (use logcounts if available)
# =========================
import scipy.sparse as sp

X = adata.layers["logcounts"] if "logcounts" in adata.layers else adata.X

if sp.issparse(X):
    X = X.toarray()

# =========================
# Labels (Malignant vs Non-Malignant)
# =========================
y = (adata.obs['cell_type'] == 'Malignant').astype(int)
y = np.array(y, dtype=int)  # ensure compatibility with CatBoost

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Models (default settings)
# =========================
models = {
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# Train & Evaluate
# =========================
for name, model in models.items():
    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Classification Report
    # =========================
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # =========================
    # Confusion Matrix
    # =========================
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-Malignant', 'Malignant'],
                yticklabels=['Non-Malignant', 'Malignant'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{name}_confusion_matrix.png", bbox_inches='tight')
    plt.close()

    # =========================
    # ROC Curve & AUC
    # =========================
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.savefig(f"{name}_ROC.png", bbox_inches='tight')
    plt.close()

    print(f"AUC: {roc_auc:.4f}")


========== RandomForest ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       293

    accuracy                           1.00       493
   macro avg       1.00      1.00      1.00       493
weighted avg       1.00      1.00      1.00       493

AUC: 1.0000

========== DecisionTree ==========

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98       200
           1       0.98      0.99      0.98       293

    accuracy                           0.98       493
   macro avg       0.98      0.98      0.98       493
weighted avg       0.98      0.98      0.98       493

AUC: 0.9799

========== XGBoost ==========


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:19:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       293

    accuracy                           1.00       493
   macro avg       1.00      1.00      1.00       493
weighted avg       1.00      1.00      1.00       493

AUC: 1.0000

========== CatBoost ==========

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       293

    accuracy                           1.00       493
   macro avg       1.00      1.00      1.00       493
weighted avg       1.00      1.00      1.00       493

AUC: 1.0000
